In [1]:
import sys
import random
import gc, argparse
import copy
import time
import glfw
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from PIL import Image
import numpy as np

import json
import os
from src.env.env_clr import RILAB_OMY_ENV
from src.controllers import load_controller 

Xlib.xauth: warning, no xauthority details available
Xlib.xauth: warning, no xauthority details available


In [2]:
# Load environment configuration
config_file_path = './configs/train_key_clr.json'
with open(config_file_path) as f:
    env_conf = json.load(f)
language_instruction = env_conf['language_instruction']
omy_env = RILAB_OMY_ENV(cfg=env_conf,
                        seed=None, 
                        action_type=env_conf['control_mode'], 
                        obs_type='eef_pose',
                        vis_mode = 'keyboard',
                        build_mjcf=False)


omy_env.reset(leader_pose = True)
# Load keyboard controller
controller = load_controller('keyboard',env_conf)
controller.reset(omy_env)


-----------------------------------------------------------------------------
name:[tabletop_env] dt:[0.002] HZ:[500]
 n_qpos:[40] n_qvel:[39] n_qacc:[39] n_ctrl:[9]
 integrator:[IMPLICITFAST]

n_body:[35]
 [0/35] [world] mass:[0.00]kg
 [1/35] [vention_rail_carriage] mass:[22.53]kg
 [2/35] [ewellix_lift_higher_link] mass:[19.17]kg
 [3/35] [ewellix_lift_middle_link] mass:[15.59]kg
 [4/35] [shoulder_link] mass:[7.37]kg
 [5/35] [upper_arm_link] mass:[13.05]kg
 [6/35] [forearm_link] mass:[3.99]kg
 [7/35] [wrist_1_link] mass:[2.10]kg
 [8/35] [wrist_2_link] mass:[1.98]kg
 [9/35] [wrist_3_link] mass:[1.56]kg
 [10/35] [finger_1_link] mass:[0.05]kg
 [11/35] [finger_2_link] mass:[0.05]kg
 [12/35] [door] mass:[0.30]kg
 [13/35] [right_latch_pull] mass:[0.10]kg
 [14/35] [left_latch_pull] mass:[0.10]kg
 [15/35] [latch_lock] mass:[0.10]kg
 [16/35] [lorge/hatch_face] mass:[19.28]kg
 [17/35] [lorge/external_rotary_wheel] mass:[0.41]kg
 [18/35] [lorge/external_rotary_handle] mass:[0.03]kg
 [19/35] [lor

You can teleop your robot with keyboard
```
---------     -----------------------
   w       ->        backward
s  a  d        left   forward   right
---------      -----------------------
In x, y plane

---------
R: Moving Up
F: Moving Down
---------
In z axis

---------
Q: Tilt left
E: Tilt right
UP: Look Upward
Down: Look Donward
Right: Turn right
Left: Turn left
---------
For rotation

---------
SPACEBAR: Toggle Gripper
--------

---------
z: reset
--------
```

In [3]:
from src.dataset.utils import make_teleoperation_dataset

make_new = True
ROOT = './dataset/clr_teleoperation_dataset'
if os.path.exists(ROOT):
    import shutil
    # print("REMOVE")
    shutil.rmtree(ROOT)
    make_new = True
if make_new:
    # print("CREATE")
    dataset = make_teleoperation_dataset(ROOT, state_dim=7)

In [4]:
NUM_trials_PER_TASK = 2
episode_id = 0

In [ ]:
while omy_env.env.is_viewer_alive():
    omy_env.step_env()
    if omy_env.env.loop_every(HZ=20):
        key_list = omy_env.env.get_key_pressed_list()
        done = omy_env.check_success()
        if done or 90 in key_list:  # 'z' key to reset
            print("END EPISODE")
            if done:
                dataset.save_episode()
                episode_id += 1
            else: 
                dataset.clear_episode_buffer()

            omy_env.reset(leader_pose = True)
            action = controller.get_action()
            eef_pose = omy_env.step(action)
        
        
        
        
        action = controller.get_action()

        eef_pose = omy_env.step(action)
        
        agent_image, wrist_image, left_scene_image, right_scene_image = omy_env.grab_image()

        images = {"agent": agent_image, 
                  "wrist": wrist_image, 
                  "left_scene": left_scene_image, 
                  "right_scene": right_scene_image}
        
        for image_label in images.keys():
            if images[image_label] is not None:
                print(image_label)
                image = Image.fromarray(images[image_label])
                # resize to 256x256
                image = image.resize((256, 256))
                image = np.array(image)
                images[image_label] = image
        
        obj_states, recp_q_poses = omy_env.get_object_pose(pad=10)
        obj_poses = np.array(obj_states['poses'])
        
        # Add frame to the dataset
        dataset.add_frame( {
                "observation.image": images["agent"],
                "observation.wrist_image": images["wrist"],
                "observation.left_scene_image": images["left_scene"],
                "observation.right_scene_image": images["right_scene"],
                "observation.state": eef_pose,
                "action": action,
                "observation.eef_pose": eef_pose,
                'env.obj_pose': np.array(obj_states['poses'],dtype=np.float32),
                "env.obj_names": ','.join(obj_states['names']),
                "env.obj_q_names": ','.join(recp_q_poses['names']),
                "env.obj_q_states": np.array(recp_q_poses['poses'],dtype=np.float32),
                "env.config_file_name": config_file_path,
                "task": language_instruction
            }, 
        )


        last_obj_poses = obj_poses
        # based on the episode_id number, get the guide line
        omy_env.render(language_instruction, guideline= f' [Num Episode: {episode_id}/{NUM_trials_PER_TASK}]')
    omy_env.env.sync_sim_wall_time()
omy_env.env.close_viewer()
dataset.finalize()